# SS7 undecodable content scan

Finds SS7 content that does **not** decode to legible text under *any*
candidate codec (gsm7/ascii/latin1/utf16) - not just whichever codec its
`dcs` value happens to map to. Two real examples that motivated this
(pasted from real data, dcs assumed 0/unremarkable):

```
020000023a130905f221332f34d4f2bf3a8a162103550407023a980610419002913574f53a0859202151029135743a182020805134233a003a00000000000000000000000000000000
030001010525000e01020c25000a083a653836525955080e2500090a980610907096878398f50525000f0100042601fa1e042602fa1e
```

**Why this needs more than `_printable_score` alone** (`ingestion/dcs_codecs.py`):
checked below (Step 2) - both examples score gsm7 `_printable_score` >= 0.97,
which looks like a clean decode, but the actual text is symbol salad
(`'$@@Δ gDèéd...'`), not language. This is the exact false-positive
mode `dcs_codecs.py`'s own module docstring warns about: GSM-7's alphabet maps
almost any byte value to *some* printable glyph, so septet-repacking random/
structured (TCAP, binary) bytes can still score near-1.0 by coincidence. ascii/
latin1/utf16 do NOT have this problem - failing outright (or scoring low) on
non-text bytes is a reliable signal for them.

**Fix used here**: trust ascii/latin1/utf16's score directly, but only accept a
gsm7 decode as "real text" if it *also* clears an `ascii_letter_ratio` floor
(fraction of characters that are plain ASCII letters/spaces - real GSM-7
SMS text is overwhelmingly that; noise reinterpreted as septets is mostly
punctuation/Greek/currency filler). Same technique
`notebooks/decode_verification.ipynb` used to catch the SS7 `dcs=4` bug.

A row is flagged **undecodable** if none of the four codecs pass under this
combined check - i.e. it doesn't look like SMS text under any interpretation
we support, real candidate for structured/binary signaling data (TCAP/MAP,
etc.), not a message a person typed.

Scans every raw SS7 file, `message_type in [2, 3]` (the only types carrying
real content - see `ingestion/ss7.py`'s module docstring), one file at a time
(not concatenated into memory), same approach as
`notebooks/ss7_regex_content_match.ipynb`.

## Step 1 - setup

Reuses the actual project decode functions (`ingestion/dcs_codecs.py`,
`ingestion/udh.py`) rather than reimplementing them, so this notebook can
never silently drift from what the real pipeline does.

In [3]:
import re
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path("..").resolve()))

from ingestion.dcs_codecs import (
    decode_ascii, decode_latin1, decode_utf16be, decode_gsm7_packed,
    _printable_score, _UNMAPPED_DCS_TRY_ORDER,
)
from ingestion.udh import parse_udh, strip_udh

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_SS7_DIR = PROJECT_ROOT / "data" / "raw" / "SS7"

CODEC_FUNCS = {
    "ascii": decode_ascii, "latin1": decode_latin1,
    "utf16": decode_utf16be, "gsm7": decode_gsm7_packed,
}
MESSAGE_TYPES_TO_CHECK = [2, 3]  # MT_request, MO - see ingestion/ss7.py

LEGIBLE_SCORE_THRESHOLD = 0.9     # same starting point as decode_verification.ipynb
GSM7_MIN_LETTER_RATIO = 0.5       # floor to trust a gsm7 decode specifically - see markdown above

USECOLS = [
    "time_stamp", "message_type", "dcs", "calling_gt", "called_gt",
    "content", "reference", "b_number",
]

raw_files = sorted(RAW_SS7_DIR.glob("*/*.csv"))
print(f"{len(raw_files)} raw SS7 files found under {RAW_SS7_DIR}")

48 raw SS7 files found under c:\Users\IshitaGodani\Documents\projects\spam-detection-prototype\data\raw\SS7


## Step 2 - scoring + classification helpers

`ascii_letter_ratio` copied from `notebooks/decode_verification.ipynb` (it
already proved out this exact fix there, for the `dcs=4` gsm7-vs-ascii
tie-break bug).

In [4]:
def ascii_letter_ratio(text):
    """Fraction of characters that are plain ASCII letters/spaces. Unlike
    _printable_score, this isn't fooled by a wrong decode that happens to
    produce exotic-but-technically-printable characters (accented letters,
    Greek letters, currency signs) - exactly what gsm7 does to noise."""
    if not text:
        return 0.0
    letters = sum(1 for ch in text if ch.isascii() and (ch.isalpha() or ch == " "))
    return letters / len(text)


def score_all_codecs(payload: bytes) -> dict:
    """Tries every codec unconditionally (ignores dcs entirely) and returns
    {codec: (text, score)} for all four."""
    return {name: (fn(payload), _printable_score(fn(payload))) for name, fn in CODEC_FUNCS.items()}


def classify_payload(payload: bytes) -> dict:
    """Decides whether `payload` looks like real text under ANY codec.

    ascii/latin1/utf16: _printable_score alone is trusted (no known false-
    positive mode for these on this data).
    gsm7: only counts as a pass if it ALSO clears GSM7_MIN_LETTER_RATIO -
    see the markdown above for why.

    Returns the best-scoring codec overall for reporting purposes even when
    that codec didn't clear its threshold (is_legible=False), so the summary
    below can still show what the "best guess" was.
    """
    scores = score_all_codecs(payload)
    gsm7_text, gsm7_score = scores["gsm7"]
    gsm7_letter_ratio = ascii_letter_ratio(gsm7_text)

    passed = {
        name: score for name, (text, score) in scores.items()
        if score >= LEGIBLE_SCORE_THRESHOLD
        and (name != "gsm7" or gsm7_letter_ratio >= GSM7_MIN_LETTER_RATIO)
    }
    is_legible = bool(passed)
    best_codec = max(passed, key=passed.get) if passed else max(scores, key=lambda n: scores[n][1])
    best_score = passed[best_codec] if passed else scores[best_codec][1]

    return {
        "is_legible": is_legible,
        "best_codec": best_codec,
        "best_score": best_score,
        "gsm7_score": gsm7_score,
        "gsm7_letter_ratio": gsm7_letter_ratio,
        "ascii_score": scores["ascii"][1],
        "latin1_score": scores["latin1"][1],
        "utf16_score": scores["utf16"][1],
    }


def classify_content_hex(content_hex) -> dict:
    empty = {"is_legible": None, "best_codec": None, "best_score": None,
              "gsm7_score": None, "gsm7_letter_ratio": None,
              "ascii_score": None, "latin1_score": None, "utf16_score": None}
    if not isinstance(content_hex, str) or not content_hex:
        return empty
    try:
        raw = bytes.fromhex(content_hex)
    except ValueError:
        return empty
    udh = parse_udh(raw)
    payload = strip_udh(raw, udh)
    if not payload:
        return empty
    return classify_payload(payload)

## Step 3 - self-test against the two motivating examples

Confirms the classifier calls both of these **undecodable** before running
the full scan.

In [5]:
EXAMPLES = [
    "020000023a130905f221332f34d4f2bf3a8a162103550407023a980610419002913574f53a0859202151029135743a182020805134233a003a00000000000000000000000000000000",
    "030001010525000e01020c25000a083a653836525955080e2500090a980610907096878398f50525000f0100042601fa1e042602fa1e",
]
for hx in EXAMPLES:
    result = classify_content_hex(hx)
    print(hx[:40] + "...")
    for k, v in result.items():
        print(f"  {k}: {v}")
    print()

020000023a130905f221332f34d4f2bf3a8a1621...
  is_legible: False
  best_codec: gsm7
  best_score: 0.9759036144578314
  gsm7_score: 0.9759036144578314
  gsm7_letter_ratio: 0.27710843373493976
  ascii_score: 0.0
  latin1_score: 0.4383561643835616
  utf16_score: 0.0

030001010525000e01020c25000a083a65383652...
  is_legible: False
  best_codec: gsm7
  best_score: 0.9836065573770492
  gsm7_score: 0.9836065573770492
  gsm7_letter_ratio: 0.29508196721311475
  ascii_score: 0.0
  latin1_score: 0.3148148148148148
  utf16_score: 0.8888888888888888



## Step 4 - full corpus scan

Streams through every raw SS7 file, one at a time, classifying every
in-scope row (`message_type in [2, 3]`, non-null `content`). Only
undecodable rows are kept in memory (with enough columns to investigate
later); everything else is just counted.

In [6]:
undecodable_chunks = []
total_rows_seen = 0
total_filtered_rows = 0        # message_type in [2, 3]
total_content_rows = 0         # of those, content is non-null
total_undecodable_rows = 0

REPORT_COLS = [
    "time_stamp", "message_type", "dcs", "calling_gt", "called_gt",
    "content", "reference", "b_number",
]

for i, path in enumerate(raw_files, start=1):
    df = pd.read_csv(path, usecols=lambda c: c in USECOLS, low_memory=False)
    total_rows_seen += len(df)

    filtered = df[df["message_type"].isin(MESSAGE_TYPES_TO_CHECK)]
    total_filtered_rows += len(filtered)

    has_content = filtered["content"].notna()
    total_content_rows += int(has_content.sum())
    scoped = filtered[has_content]
    if scoped.empty:
        continue

    classified = scoped["content"].apply(classify_content_hex).apply(pd.Series)
    undecodable_mask = classified["is_legible"] == False  # noqa: E712 (None means unparseable, not undecodable)
    total_undecodable_rows += int(undecodable_mask.sum())

    if undecodable_mask.any():
        rows = pd.concat(
            [scoped.loc[undecodable_mask, REPORT_COLS].reset_index(drop=True),
             classified.loc[undecodable_mask].reset_index(drop=True)],
            axis=1,
        )
        undecodable_chunks.append(rows)

    if i % 10 == 0 or i == len(raw_files):
        print(f"[{i}/{len(raw_files)}] {path.name}: {len(df)} rows, "
              f"{len(scoped)} scoped, {total_undecodable_rows} undecodable so far")

undecodable = (
    pd.concat(undecodable_chunks, ignore_index=True)
    if undecodable_chunks else
    pd.DataFrame(columns=REPORT_COLS + ["is_legible", "best_codec", "best_score",
                                          "gsm7_score", "gsm7_letter_ratio",
                                          "ascii_score", "latin1_score", "utf16_score"])
)

print(f"\ntotal rows scanned: {total_rows_seen:,}")
print(f"rows in scope (message_type in {MESSAGE_TYPES_TO_CHECK}): {total_filtered_rows:,}")
print(f"in-scope rows with content: {total_content_rows:,}")
print(f"undecodable under any codec: {total_undecodable_rows:,} "
      f"({total_undecodable_rows / total_content_rows:.2%} of content rows)" if total_content_rows else "")

[10/48] stg_ss7_20260802_0900.csv: 266177 rows, 57817 scoped, 92254 undecodable so far
[20/48] stg_ss7_20260802_1900.csv: 339103 rows, 76637 scoped, 252248 undecodable so far
[30/48] stg_ss7_20260803_0500.csv: 157936 rows, 28451 scoped, 342392 undecodable so far
[40/48] stg_ss7_20260803_1500.csv: 542639 rows, 157551 scoped, 498753 undecodable so far
[48/48] stg_ss7_20260803_2300.csv: 261057 rows, 53594 scoped, 598709 undecodable so far

total rows scanned: 14,685,436
rows in scope (message_type in [2, 3]): 3,417,425
in-scope rows with content: 3,416,308
undecodable under any codec: 598,709 (17.53% of content rows)


## Summary tables

In [7]:
print("undecodable count by message_type:")
undecodable["message_type"].value_counts().sort_index()

undecodable count by message_type:


message_type
2     69564
3    529145
Name: count, dtype: int64

In [8]:
print("undecodable count by dcs:")
undecodable["dcs"].value_counts().sort_index()

undecodable count by dcs:


dcs
0      227008
1         281
3          90
4      336436
8       27826
16         41
26          6
241      4001
245       841
246      2179
Name: count, dtype: int64

In [9]:
print("best_codec breakdown (the closest guess, even though it failed the legibility bar):")
undecodable["best_codec"].value_counts()

best_codec breakdown (the closest guess, even though it failed the legibility bar):


best_codec
gsm7      597299
utf16        758
latin1       519
ascii        133
Name: count, dtype: int64

In [10]:
calling_gt_counts = undecodable["calling_gt"].value_counts()
print(f"distinct callingGT among undecodable rows: {calling_gt_counts.shape[0]}")
calling_gt_counts

distinct callingGT among undecodable rows: 509


calling_gt
60120000664     180190
60120000663     111483
60120000665     105282
60120000060      44811
60120000061      41385
                 ...  
905329372200         1
919847299934         1
8613090868           1
353830000005         1
97333811012          1
Name: count, Length: 509, dtype: int64

In [11]:
called_gt_counts = undecodable["called_gt"].value_counts()
print(f"distinct calledGT among undecodable rows: {called_gt_counts.shape[0]}")
called_gt_counts

distinct calledGT among undecodable rows: 13


called_gt
60120000015        529140
60120000062         23057
60120000061         21072
60120000060         18267
60120001128          4757
60120000664          1111
60120000665           717
60120000663           583
601200000152255         1
601200000152222         1
601200000154            1
601200000152            1
6012000001500           1
Name: count, dtype: int64

## Full undecodable-row list

In [ ]:
undecodable

In [12]:
out_path = PROJECT_ROOT / "notebooks" / "ss7_undecodable_content.csv"
undecodable.to_csv(out_path, index=False)
print(f"wrote {len(undecodable)} rows to {out_path}")

wrote 598709 rows to c:\Users\IshitaGodani\Documents\projects\spam-detection-prototype\notebooks\ss7_undecodable_content.csv


## Scratch cell - inspect one hex string by hand

Paste a `content` hex string below and run this cell to see every codec's
score/text side by side, and whether it clears the legibility bar.

In [13]:
# --- Edit this line, then run the cell ---
content_hex = "020000023a130905f221332f34d4f2bf3a8a162103550407023a980610419002913574f53a0859202151029135743a182020805134233a003a00000000000000000000000000000000"
# -------------------------------------------

raw = bytes.fromhex(content_hex)
udh = parse_udh(raw)
payload = strip_udh(raw, udh)
scores = score_all_codecs(payload)
result = classify_payload(payload)

print(f"raw bytes ({len(raw)}): {raw!r}")
print(f"udh.present: {udh.present}")
print(f"payload (post-UDH, {len(payload)} bytes): {payload!r}\n")
for name, (text, score) in scores.items():
    print(f"{name:8s} score={score:.3f}  text={text!r}")
print()
for k, v in result.items():
    print(f"{k}: {v}")

raw bytes (73): b'\x02\x00\x00\x02:\x13\t\x05\xf2!3/4\xd4\xf2\xbf:\x8a\x16!\x03U\x04\x07\x02:\x98\x06\x10A\x90\x02\x915t\xf5:\x08Y !Q\x02\x915t:\x18  \x80Q4#:\x00:\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'
udh.present: False
payload (post-UDH, 73 bytes): b'\x02\x00\x00\x02:\x13\t\x05\xf2!3/4\xd4\xf2\xbf:\x8a\x16!\x03U\x04\x07\x02:\x98\x06\x10A\x90\x02\x915t\xf5:\x08Y !Q\x02\x915t:\x18  \x80Q4#:\x00:\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'

ascii    score=0.000  text=None
latin1   score=0.438  text='\x02\x00\x00\x02:\x13\t\x05ò!3/4Ôò¿:\x8a\x16!\x03U\x04\x07\x02:\x98\x06\x10A\x90\x02\x915tõ:\x08Y !Q\x02\x915t:\x18  \x80Q4#:\x00:\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'
utf16    score=0.000  text=None
gsm7     score=0.976  text='$@@Δ gDèédìΘsé\rjràjQh"H£UòÆΔ ì&¥Δ$AΛΔ2\r:uu Hé¤H($"V!\'ìùΔ @F"3DÅ@:@@@@@@@@@@@@@@@@@@'

is_legible: False
best_codec: gsm7
best_score: 0.9759036144578314
gsm7_score: 0.9759036144578314
gsm7_l